In [6]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [7]:
df = pd.read_csv("dice_com-job_us_sample.csv.csv")

df.head()

,advertiserurl,company,employmenttype_jobstatus,jobdescription,jobid,joblocation_address,jobtitle,postdate
0,https://www.dice.com/jobs/detail/AUTOMATION-TE...,"Digital Intelligence Systems, LLC","C2H Corp-To-Corp, C2H Independent, C2H W2, 3 M...",Looking for Selenium engineers...must have sol...,Dice Id : 10110693,"Atlanta, GA",AUTOMATION TEST ENGINEER,1 hour ago
1,https://www.dice.com/jobs/detail/Information-S...,University of Chicago/IT Services,Full Time,The University of Chicago has a rapidly growin...,Dice Id : 10114469,"Chicago, IL",Information Security Engineer,1 week ago
2,https://www.dice.com/jobs/detail/Business-Solu...,"Galaxy Systems, Inc.",Full Time,"GalaxE.SolutionsEvery day, our solutions affec...",Dice Id : CXGALXYS,"Schaumburg, IL",Business Solutions Architect,2 weeks ago
3,https://www.dice.com/jobs/detail/Java-Develope...,TransTech LLC,Full Time,Java DeveloperFull-time/direct-hireBolingbrook...,Dice Id : 10113627,"Bolingbrook, IL","Java Developer (mid level)- FT- GREAT culture,...",2 weeks ago
4,https://www.dice.com/jobs/detail/DevOps-Engine...,Matrix Resources,Full Time,Midtown based high tech firm has an immediate ...,Dice Id : matrixga,"Atlanta, GA",DevOps Engineer,48 minutes ago


In [8]:
print(df.columns)

Index(['advertiserurl', 'company', 'employmenttype_jobstatus',
       'jobdescription', 'jobid', 'joblocation_address', 'jobtitle',
       'postdate'],
      dtype='str')


In [9]:
df = df[['jobtitle', 'jobdescription']]
df.head()

,jobtitle,jobdescription
0,AUTOMATION TEST ENGINEER,Looking for Selenium engineers...must have sol...
1,Information Security Engineer,The University of Chicago has a rapidly growin...
2,Business Solutions Architect,"GalaxE.SolutionsEvery day, our solutions affec..."
3,"Java Developer (mid level)- FT- GREAT culture,...",Java DeveloperFull-time/direct-hireBolingbrook...
4,DevOps Engineer,Midtown based high tech firm has an immediate ...


In [10]:
df = df[['jobtitle','jobdescription']]

In [11]:
df = df.dropna()

print(df.shape)

(9000, 2)


In [12]:


top_jobs = df['jobtitle'].value_counts().head(20).index

df = df[df['jobtitle'].isin(top_jobs)]

print("Dataset Shape:", df.shape)

print("\nTop 20 Job Titles:")

print(df['jobtitle'].value_counts())

Dataset Shape: (581, 2)

Top 20 Job Titles:
jobtitle
Java Developer                   75
Network Engineer                 54
Project Manager                  53
Business Analyst                 45
Software Engineer                45
Software Development Engineer    28
Systems Engineer                 26
.Net Developer                   25
Web Developer                    24
Systems Administrator            24
Senior Java Developer            22
Data Analyst                     21
Software Developer               19
DevOps Engineer                  18
Front End Developer              18
Data Scientist                   18
Android Developer                18
.NET Developer                   18
Business Systems Analyst         15
Senior Network Engineer          15
Name: count, dtype: int64


In [13]:
X = df['jobdescription']
y = df['jobtitle']

In [14]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Total Classes:", len(label_encoder.classes_))

Total Classes: 20


In [15]:
tokenizer = Tokenizer(num_words=10000)

tokenizer.fit_on_texts(X)

sequences = tokenizer.texts_to_sequences(X)

In [16]:
max_len = 200

X_pad = pad_sequences(
    sequences,
    maxlen=max_len,
    padding='post'
)

print(X_pad.shape)

(581, 200)


In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X_pad,
    y_encoded,
    test_size=0.2,
    random_state=42
)

In [18]:
model = Sequential([
    Embedding(10000, 64, input_length=max_len),
    GlobalAveragePooling1D(),
    Dense(128, activation='relu'),
    Dense(len(label_encoder.classes_), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

c:\Users\navee\Desktop\Data_Scientist\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [19]:
history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1
)

Epoch 1/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.1164 - loss: 2.9867 - val_accuracy: 0.1624 - val_loss: 2.9562
Epoch 2/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.1207 - loss: 2.9446 - val_accuracy: 0.1624 - val_loss: 2.8892
Epoch 3/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.1207 - loss: 2.8891 - val_accuracy: 0.1624 - val_loss: 2.8236
Epoch 4/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1207 - loss: 2.8427 - val_accuracy: 0.1624 - val_loss: 2.7918
Epoch 5/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.1659 - loss: 2.8056 - val_accuracy: 0.2051 - val_loss: 2.7728
Epoch 6/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.1767 - loss: 2.7669 - val_accuracy: 0.1880 - val_loss: 2.7534
Epoch 7/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.2112 - loss: 2.7156 - val_accuracy: 0.2308 - val_loss: 2.7230
Epoch 8/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.2349 - loss: 2.6572 - val_accuracy: 0.2564 - v

In [20]:
loss, acc = model.evaluate(X_test, y_test)

print("Test Accuracy:", acc * 100)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4274 - loss: 1.8473 
Test Accuracy: 42.73504316806793


In [21]:
model.save("job_model.h5") 

print("Model Saved Successfully")

Model Saved Successfully


In [22]:
with open("tokenizer.pkl","wb") as f:
    pickle.dump(tokenizer,f)

print("Tokenizer Saved")

Tokenizer Saved


In [23]:
with open("label_encoder.pkl","wb") as f:
    pickle.dump(label_encoder,f)

print("Label Encoder Saved")

Label Encoder Saved


In [24]:
sample = """
Python developer with machine learning,
SQL, Pandas, NumPy and Deep Learning skills
"""

seq = tokenizer.texts_to_sequences([sample])

pad = pad_sequences(seq,maxlen=max_len,padding='post')

pred = model.predict(pad)

job = label_encoder.inverse_transform([np.argmax(pred)])

print("Recommended Job:", job[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
Recommended Job: Java Developer
